In [3]:
# LAB 3: Prompt Engineering Patterns & Conversation Memory
# Google Colab + LangChain + Hugging Face

# STEP 1: Install required libraries
!pip install -q langchain langchain-community transformers accelerate sentencepiece

import requests
print(requests.__version__)

2.32.5


In [5]:
# STEP 2: Load Hugging Face LLM (FLAN-T5)

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [6]:
# STEP 3: Import PromptTemplate and Memory components

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Try importing memory - handle different LangChain versions
try:
    from langchain.memory import ConversationBufferMemory
    from langchain.chains import ConversationChain
except ImportError:
    try:
        from langchain_community.memory import ConversationBufferMemory
        from langchain.chains import ConversationChain
    except ImportError:
        print("Note: Using alternative memory approach")
        ConversationBufferMemory = None
        ConversationChain = None

Note: Using alternative memory approach


In [7]:
# STEP 4: Demonstration WITHOUT Memory (Problem)
simple_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the question:\n{question}"
)

print("=== WITHOUT MEMORY ===")
print("Q: What is Artificial Intelligence?")
print("A:", llm.invoke(simple_prompt.format(
    question="What is Artificial Intelligence?"
)))

print("\nQ: Where is it used?")
print("A:", llm.invoke(simple_prompt.format(
    question="Where is it used?"
)))

=== WITHOUT MEMORY ===
Q: What is Artificial Intelligence?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Answer the question:
What is Artificial Intelligence?

Q: Where is it used?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Answer the question:
Where is it used?


In [8]:
conversation_history = []

def ask_with_memory(question):
    """Ask a question with conversation memory"""
    # Build context from conversation history
    history_context = ""
    if conversation_history:
        history_context = "\n\nPrevious conversation:\n"
        for i, (q, a) in enumerate(conversation_history, 1):
            history_context += f"Q{i}: {q}\nA{i}: {a}\n"

    # Create prompt with history
    prompt_text = f"""You are a helpful assistant. Use the conversation history to answer questions.{history_context}

Current question: {question}

Answer:"""

    # Get response
    response = llm.invoke(prompt_text)

    # Store in memory
    conversation_history.append((question, response))

    return response

# Clear previous conversation
conversation_history = []

print("=== WITH MEMORY ===")
print("Q: Who is Sundar Pichai?")
response1 = ask_with_memory("Who is Sundar Pichai?")
print("A:", response1)

print("\nQ: Which company is he the CEO of?")
response2 = ask_with_memory("Which company is he the CEO of?")
print("A:", response2)

print("\nQ: Where is that company headquartered?")
response3 = ask_with_memory("Where is that company headquartered?")
print("A:", response3)


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== WITH MEMORY ===
Q: Who is Sundar Pichai?
A: You are a helpful assistant. Use the conversation history to answer questions.

Current question: Who is Sundar Pichai?

Answer:

Q: Which company is he the CEO of?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: You are a helpful assistant. Use the conversation history to answer questions.

Previous conversation:
Q1: Who is Sundar Pichai?
A1: You are a helpful assistant. Use the conversation history to answer questions.

Current question: Who is Sundar Pichai?

Answer:


Current question: Which company is he the CEO of?

Answer:

Q: Where is that company headquartered?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: You are a helpful assistant. Use the conversation history to answer questions.

Previous conversation:
Q1: Who is Sundar Pichai?
A1: You are a helpful assistant. Use the conversation history to answer questions.

Current question: Who is Sundar Pichai?

Answer:
Q2: Which company is he the CEO of?
A2: You are a helpful assistant. Use the conversation history to answer questions.

Previous conversation:
Q1: Who is Sundar Pichai?
A1: You are a helpful assistant. Use the conversation history to answer questions.

Current question: Who is Sundar Pichai?

Answer:


Current question: Which company is he the CEO of?

Answer:


Current question: Where is that company headquartered?

Answer:
